<a href="https://colab.research.google.com/github/DhimanTarafdar/simple-gk-answring-system-using-rnn/blob/main/simple_gk_answring_system_code_explain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

## ১. Dataset লোড করা

```python
!git clone https://github.com/tajuar-akash-hub/Datasets
```

```python
import pandas as pd

df = pd.read_csv("/content/Datasets/gk_qna_dataset.csv")
df
```

```python
df['question']       # শুধু question column দেখা
df['question'][0]    # প্রথম প্রশ্নটি দেখা
```

### 🔍 ব্যাখ্যা

| লাইন | কাজ |
|------|-----|
| `!git clone ...` | GitHub থেকে dataset সহ repository clone করে Google Colab-এ নিয়ে আসে |
| `import pandas as pd` | Data manipulation-এর জন্য pandas library import করা হয় |
| `pd.read_csv(...)` | CSV ফাইলটি পড়ে একটি DataFrame (টেবিল) তৈরি করে `df` ভেরিয়েবলে রাখা হয় |
| `df['question']` | DataFrame-এর শুধু "question" column-টি বের করে দেখা হয় |
| `df['question'][0]` | ০ নম্বর index-এর (প্রথম) প্রশ্নটি দেখা হয় |

**Dataset Structure:** CSV-তে দুটি column আছে — `question` এবং `answer`। প্রতিটি row একটি GK প্রশ্ন এবং তার উত্তর।

---

## ২. Tokenization (টোকেনাইজেশন)

```python
import re

def tokenize(text):

    # ধাপ ১: সব অক্ষর lowercase করা
    text = text.lower()

    # ধাপ ২: কোটেশন চিহ্ন সরানো (" এবং ')
    text = re.sub(r"[\"']", "", text)

    # ধাপ ৩: শুধু অক্ষর (a-z) ও সংখ্যা (0-9) ছাড়া বাকি সব চিহ্ন সরানো
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # ধাপ ৪: একাধিক space কে একটি space বানানো
    text = re.sub(r"\s+", " ", text).strip()

    # ধাপ ৫: শব্দে ভাগ করা (split)
    tokens = text.split()

    return tokens
```

```python
print(tokenize(df['question'][0]))
# উদাহরণ আউটপুট: ['what', 'is', 'the', 'capital', 'of', 'france']
```

### 🔍 ব্যাখ্যা

Tokenization মানে হলো একটি বড় text-কে ছোট ছোট শব্দে (token) ভেঙে দেওয়া। Machine Learning মডেল সরাসরি text বোঝে না, তাই আগে text পরিষ্কার করে শব্দে ভাগ করতে হয়।

| ধাপ | Code | উদ্দেশ্য |
|-----|------|----------|
| ১ | `text.lower()` | "What" এবং "what" কে একই শব্দ হিসেবে গণ্য করতে সব lowercase করা হয় |
| ২ | `re.sub(r"[\"']", ...)` | `"Paris"` বা `'Paris'` থেকে কোটেশন সরানো |
| ৩ | `re.sub(r"[^a-z0-9\s]", ...)` | `?`, `.`, `!` এই ধরনের punctuation সরানো, কারণ এগুলো অর্থ বহন করে না |
| ৪ | `re.sub(r"\s+", ...)` | "hello&nbsp;&nbsp;&nbsp;world" → "hello world" — অতিরিক্ত space পরিষ্কার |
| ৫ | `text.split()` | "what is france" → `['what', 'is', 'france']` — list of words |

**`re` module কী?** Python-এর Regular Expression (regex) library, যা text pattern matching ও replacement করতে ব্যবহার হয়।

---

## ৩. Vocabulary (শব্দভাণ্ডার) তৈরি করা

```python
# UNK মানে Unknown — যে শব্দ vocab-এ নেই তাকে 0 দিয়ে represent করা হবে
vocab = {'<UNK>': 0}
```

```python
def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer   = tokenize(row['answer'])

    # question ও answer-এর সব token একসাথে মেলানো
    merged_tokens = tokenized_question + tokenized_answer

    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)   # নতুন শব্দকে পরবর্তী index দাও
```

```python
df.apply(build_vocab, axis=1)   # প্রতিটি row-এর উপর build_vocab চালানো

print(vocab)          # পুরো vocab dictionary দেখা
print(len(vocab))     # মোট কতটি unique শব্দ আছে
print(vocab['what'])  # 'what' শব্দের index কত
```

### 🔍 ব্যাখ্যা

**Vocabulary কেন দরকার?** Neural Network সংখ্যা ছাড়া কাজ করতে পারে না। তাই প্রতিটি অনন্য শব্দকে একটি unique number (index) দিতে হয়।

| Code | কাজ |
|------|-----|
| `vocab = {'<UNK>': 0}` | Dictionary শুরু করা হয় — `<UNK>` মানে অজানা শব্দ, index = 0 |
| `tokenize(row['question'])` | প্রতিটি প্রশ্নকে token করা |
| `merged_tokens = q_tokens + a_tokens` | প্রশ্ন ও উত্তর দুটোর শব্দ একসাথে ধরা হয় |
| `if token not in vocab` | যদি শব্দটি আগে না থাকে, তাহলে নতুন index দাও |
| `vocab[token] = len(vocab)` | সেই মুহূর্তে vocab-এ যতটি শব্দ আছে, সেটাই নতুন শব্দের index হয় |
| `df.apply(build_vocab, axis=1)` | DataFrame-এর প্রতিটি row-এর উপর function চালানো হয় |

**উদাহরণ:** "What is France?" → `{'<UNK>':0, 'what':1, 'is':2, 'france':3, ...}`

---